# Lab 1 · B. Azure AI Document Intelligence con data sintética
**Proyecto Integrador:** Copiloto Cognitivo de Asistencia Bancaria · **Autor:** jorgecr1991

Se reproducen las pruebas de la sesión de nivelación (`nb_DocIntell.ipynb`) sobre documentos **100% sintéticos** del proyecto:

| Archivo | Contenido sintético | Modelo |
| --- | --- | --- |
| `data/ficha_tarjetas_credito_banco_andino.pdf` | Ficha de producto y guía de atención de tarjetas de crédito “Cóndor” de un banco ficticio (2 páginas, 4 tablas, procedimientos) | `prebuilt-layout` |
| `data/factura_sintetica_plasticos_andes.pdf` | Factura ficticia de un proveedor de plásticos al banco ficticio (1 página) | `prebuilt-invoice` |

Los PDF se generaron con IA y el script `scripts/generar_pdfs_sinteticos.py`. Ningún dato corresponde a clientes, colaboradores ni productos reales.

## Paso 2. Configurar los clientes con la identidad ya validada

In [1]:
# Carga de configuración (solo endpoints y nombres; NUNCA API keys)
import os
from pathlib import Path
from dotenv import load_dotenv

CANDIDATOS_ENV = [
    Path(".env"),  # labs/lab01/.env (opcional, propio del laboratorio)
    Path("../../8.- Material Clases/1.- Nivelacion/2.- Cognitive AI Foundations/Notebooks/.env"),  # .env del curso
]
ENV_USADO = next((p for p in CANDIDATOS_ENV if p.exists()), None)
if ENV_USADO is None:
    raise FileNotFoundError("No se encontró un .env. Copia .env.example a .env y completa los endpoints.")
load_dotenv(ENV_USADO, override=True)
print("Configuración cargada desde:", ENV_USADO.resolve())

# Control de seguridad: el .env no debe contener claves ni secretos con valor
claves = []
for linea in ENV_USADO.read_text(encoding="utf-8").splitlines():
    if "=" in linea and not linea.strip().startswith("#"):
        nombre, valor = linea.split("=", 1)
        if any(t in nombre.upper() for t in ("KEY", "SECRET", "PASSWORD", "TOKEN")) and valor.strip().strip('"'):
            claves.append(nombre.strip())
assert not claves, f"El .env contiene variables tipo clave: {claves}. Elimínalas: la autenticación es con az login."
print("OK: no hay API keys en la configuración; la autenticación será con Microsoft Entra ID (az login).")

Configuración cargada desde: C:\Users\jorge\Documents\Credicorp_Plan de Trabajo_Contenido formativo\8.- Material Clases\1.- Nivelacion\2.- Cognitive AI Foundations\Notebooks\.env
OK: no hay API keys en la configuración; la autenticación será con Microsoft Entra ID (az login).


In [2]:
import json
from datetime import date
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient, DocumentIntelligenceAdministrationClient
from azure.ai.documentintelligence.models import DocumentContentFormat
from azure.core.exceptions import HttpResponseError

endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")
credential = DefaultAzureCredential()          # usa la sesión de `az login`
client = DocumentIntelligenceClient(endpoint=endpoint, credential=credential)
admin_client = DocumentIntelligenceAdministrationClient(endpoint=endpoint, credential=credential)

RUTA_FICHA = Path("data/ficha_tarjetas_credito_banco_andino.pdf")
RUTA_FACTURA = Path("data/factura_sintetica_plasticos_andes.pdf")
SALIDAS = Path("outputs"); SALIDAS.mkdir(exist_ok=True)
assert RUTA_FICHA.exists() and RUTA_FACTURA.exists(), "Ejecuta primero scripts/generar_pdfs_sinteticos.py"
print("Clientes configurados con DefaultAzureCredential para:", endpoint)

Clientes configurados con DefaultAzureCredential para: https://docintellntt.cognitiveservices.azure.com/


Función de apoyo reutilizada del notebook de clase (convierte el resultado a Markdown con tablas).

In [3]:
# Resultado como markdown
def resultado_a_markdown(resultado, titulo="Documento procesado"):
    """
    Convierte el resultado de Azure AI Document Intelligence
    (por ejemplo, prebuilt-layout) a Markdown, incluyendo tablas.

    Args:
        resultado: Resultado de poller.result().
        titulo: Título del documento.

    Returns:
        str: Documento Markdown.
    """
    lineas_md = [
        f"# {titulo}",
        "",
        f"**Modelo utilizado:** `{resultado.model_id}`",
        f"**Número de páginas:** {len(resultado.pages or [])}",
        f"**Tablas detectadas:** {len(resultado.tables or [])}",
        "",
        "---",
        "",
    ]

    def limpiar(texto):
        """Evita que caracteres del contenido rompan la tabla Markdown."""
        if not texto:
            return ""
        return (
            str(texto)
            .replace("|", r"\|")
            .replace("\r", " ")
            .replace("\n", " ")
            .strip()
        )

    def tabla_a_markdown(tabla):
        """Convierte una tabla de Document Intelligence a Markdown."""
        filas = [
            [""] * tabla.column_count
            for _ in range(tabla.row_count)
        ]

        for celda in tabla.cells:
            filas[celda.row_index][celda.column_index] = limpiar(
                celda.content
            )

        # Elimina filas completamente vacías al principio y al final.
        while filas and not any(filas[0]):
            filas.pop(0)

        while filas and not any(filas[-1]):
            filas.pop()

        if not filas:
            return ""

        # Completa las filas para asegurar el mismo número de columnas.
        ancho = max(len(fila) for fila in filas)
        filas = [fila + [""] * (ancho - len(fila)) for fila in filas]

        salida = []
        salida.append("| " + " | ".join(filas[0]) + " |")
        salida.append("| " + " | ".join(["---"] * ancho) + " |")

        for fila in filas[1:]:
            salida.append("| " + " | ".join(fila) + " |")

        return "\n".join(salida)

    # Organiza las tablas por página de aparición, usando bounding_regions.
    tablas_por_pagina = {}

    for indice, tabla in enumerate(resultado.tables or [], start=1):
        regiones = getattr(tabla, "bounding_regions", None) or []

        paginas_tabla = {
            region.page_number for region in regiones
        } or {None}

        for numero_pagina in paginas_tabla:
            tablas_por_pagina.setdefault(numero_pagina, []).append(
                (indice, tabla)
            )

    for pagina in resultado.pages or []:
        numero_pagina = pagina.page_number

        lineas_md.extend([
            f"## Página {numero_pagina}",
            "",
        ])

        # Obtiene las tablas asociadas a esta página.
        tablas_pagina = tablas_por_pagina.get(numero_pagina, [])
        tablas_pagina_ids = {indice for indice, _ in tablas_pagina}

        # Texto detectado en la página.
        if pagina.lines:
            for linea in pagina.lines:
                texto = limpiar(linea.content)
                if texto:
                    lineas_md.extend([texto, ""])

        else:
            lineas_md.extend([
                "*No se detectó texto en esta página.*",
                "",
            ])

        # Muestra las tablas estructuradas detectadas en la página.
        if tablas_pagina:
            lineas_md.extend([
                f"### Tablas de la página {numero_pagina}",
                "",
            ])

            for indice, tabla in tablas_pagina:
                # Evita duplicar una tabla que aparece en varias páginas.
                if indice in getattr(
                    resultado_a_markdown, "_tablas_mostradas", set()
                ):
                    continue

                lineas_md.extend([
                    f"**Tabla {indice}**",
                    "",
                    tabla_a_markdown(tabla),
                    "",
                ])

                # El seguimiento real se gestiona fuera de la función.
                # Se actualiza el conjunto local al final de cada tabla.

    return "\n".join(lineas_md)

## Paso 3. Listar los modelos disponibles

In [4]:
try:
    modelos = list(admin_client.list_models())
except HttpResponseError as e:
    raise SystemExit(f"Error HTTP {e.status_code}: {e.message} → registrar y comunicar al instructor (no modificar roles).")

prebuilt = [m for m in modelos if m.model_id.startswith("prebuilt-")]
custom = [m for m in modelos if not m.model_id.startswith("prebuilt-")]
print(f"Total de modelos: {len(modelos)} | prebuilt: {len(prebuilt)} | personalizados: {len(custom)}\n")
for m in prebuilt:
    print(f"- {m.model_id:<32} {m.description or ''}")
for requerido in ("prebuilt-layout", "prebuilt-invoice"):
    print(f"\n¿{requerido} disponible?", "Sí" if any(m.model_id == requerido for m in modelos) else "NO")

Total de modelos: 67 | prebuilt: 66 | personalizados: 1

- prebuilt-bankStatement.us        Extract key information from US bank statement.
- prebuilt-check.us                Extract key information from United States check.
- prebuilt-contract                Extract key information from legal contracts.
- prebuilt-creditCard              Extract key information from credit card.
- prebuilt-healthInsuranceCard.us  Extract key information from US health insurance cards.
- prebuilt-idDocument              Extract key information from passports and ID cards.
- prebuilt-invoice                 Extract key information from invoices.
- prebuilt-layout                  Extract text and layout information from documents.
- prebuilt-marriageCertificate.us  Extract key information from US marriage certificate.
- prebuilt-mortgage.us.1003        Extract key information from US Uniform Residential Loan Application (Form 1003).
- prebuilt-mortgage.us.1004        Extract key information from US Unif

## Paso 4. Probar `prebuilt-layout` sobre la ficha de producto
Se solicita además la salida en **Markdown**, que conserva títulos y tablas (útil para RAG).

In [5]:
with open(RUTA_FICHA, "rb") as archivo:
    poller = client.begin_analyze_document(
        model_id="prebuilt-layout",
        body=archivo,
        content_type="application/octet-stream",
        output_content_format=DocumentContentFormat.MARKDOWN,
    )
layout = poller.result()

roles = {}
for p in layout.paragraphs or []:
    roles[p.role or "texto"] = roles.get(p.role or "texto", 0) + 1

print("Análisis completado · modelo:", layout.model_id)
print("Páginas       :", len(layout.pages or []))
print("Líneas        :", sum(len(pg.lines or []) for pg in layout.pages or []))
print("Palabras      :", sum(len(pg.words or []) for pg in layout.pages or []))
print("Párrafos      :", len(layout.paragraphs or []), "→ por rol:", roles)
print("Secciones     :", len(layout.sections or []))
print("Tablas        :", len(layout.tables or []))
for i, t in enumerate(layout.tables or [], start=1):
    pagina = t.bounding_regions[0].page_number if t.bounding_regions else "?"
    print(f"  Tabla {i}: {t.row_count} filas x {t.column_count} columnas (página {pagina})")

Análisis completado · modelo: prebuilt-layout
Páginas       : 2
Líneas        : 152
Palabras      : 695
Párrafos      : 127 → por rol: {<ParagraphRole.TITLE: 'title'>: 1, 'texto': 114, <ParagraphRole.SECTION_HEADING: 'sectionHeading'>: 8, <ParagraphRole.PAGE_FOOTER: 'pageFooter'>: 2, <ParagraphRole.PAGE_NUMBER: 'pageNumber'>: 2}
Secciones     : 9
Tablas        : 4
  Tabla 1: 5 filas x 4 columnas (página 1)
  Tabla 2: 8 filas x 5 columnas (página 1)
  Tabla 3: 6 filas x 4 columnas (página 2)
  Tabla 4: 5 filas x 3 columnas (página 2)


In [6]:
# Títulos y encabezados de sección detectados
for p in layout.paragraphs or []:
    if p.role in ("title", "sectionHeading"):
        print(f"[{p.role}] {p.content}")

[ParagraphRole.TITLE] Banco Andino Ficticio S.A. Ficha de producto y guía de atención: Tarjetas de crédito Cóndor
[ParagraphRole.SECTION_HEADING] 1. Propósito y alcance
[ParagraphRole.SECTION_HEADING] 2. Catálogo de tarjetas
[ParagraphRole.SECTION_HEADING] 3. Costos y tarifario referencial
[ParagraphRole.SECTION_HEADING] 4. Requisitos de solicitud
[ParagraphRole.SECTION_HEADING] 5. Procedimiento de solicitud (canal agencia)
[ParagraphRole.SECTION_HEADING] 6. Bloqueo por pérdida o robo
[ParagraphRole.SECTION_HEADING] 7. Reclamos y plazos de atención
[ParagraphRole.SECTION_HEADING] 8. Restricciones y buenas prácticas


In [7]:
# Vista con la función de clase (texto por página + tablas estructuradas)
print(resultado_a_markdown(layout, titulo="Ficha de tarjetas de crédito Cóndor (sintética)"))

# Ficha de tarjetas de crédito Cóndor (sintética)

**Modelo utilizado:** `prebuilt-layout`
**Número de páginas:** 2
**Tablas detectadas:** 4

---

## Página 1

Banco Andino Ficticio S.A.

Ficha de producto y guía de atención: Tarjetas de crédito

Cóndor

Código de documento: BAF-TC-GUIA-007 · Versión 3.2 · Vigencia: 01/10/2026 - 31/03/2027 · Área responsable: Gestión de Productos

de Tarjetas

1. Propósito y alcance

Esta guía orienta a los especialistas de negocio en la atención de consultas sobre la familia de tarjetas de crédito

Cóndor: características, costos, requisitos, procedimientos de solicitud, bloqueo y reclamos. Las tasas y

comisiones vigentes deben confirmarse siempre en el sistema de características de productos; ante una

diferencia, prevalece el sistema.

2. Catálogo de tarjetas

Tarjeta

Segmento objetivo

Línea de crédito (S/)

Programa de beneficios

Cóndor Clásica

Primer tarjeta, ingresos

desde S/ 1,500

1,000 - 8,000

1 punto Cumbre por cada S/ 10

Cóndor Oro



## Paso 5. Probar `prebuilt-invoice` sobre la factura sintética

In [8]:
with open(RUTA_FACTURA, "rb") as archivo:
    poller = client.begin_analyze_document(
        model_id="prebuilt-invoice",
        body=archivo,
        content_type="application/octet-stream",
    )
invoices = poller.result()

print("Modelo:", invoices.model_id, "| páginas:", len(invoices.pages or []),
      "| tablas:", len(invoices.tables or []), "| documentos:", len(invoices.documents or []))
print("\nTexto detectado (primeros 600 caracteres):\n", (invoices.content or "")[:600])

Modelo: prebuilt-invoice | páginas: 1 | tablas: 2 | documentos: 1

Texto detectado (primeros 600 caracteres):
 Plásticos y Seguridad Andes S.A.C.
FACTURA ELECTRÓNICA
RUC: 20999888771
Nº F001-004587
Av. Los Inventados 742, Lima, Perú
Fecha de emisión: 15/10/2026
facturacion@plasticos-andes.example
Fecha de vencimiento: 14/11/2026
Cliente: Banco Andino Ficticio S.A.
RUC: 20111222333
Dirección: Jr. Imaginario 1200, San Isidro, Lima
Orden de compra: OC-2026-0931 · Moneda: Soles (PEN) · Condición: Crédito 30 días
Ítem
Descripción
Cant.
Valor unit. (S/)
Importe (S/)
1
Plástico tarjeta Cóndor Clásica con chip EMV
5,000
3.20
16,000.00
2
Plástico tarjeta Cóndor Oro con chip EMV y contactless
2,000
4.10
8,200.00


In [9]:
def valor_campo(campo):
    """Devuelve el valor más legible de un campo de Document Intelligence."""
    if campo is None:
        return None
    moneda = getattr(campo, "value_currency", None)
    if moneda is not None:
        return f"{moneda.currency_symbol or moneda.currency_code or ''} {moneda.amount}".strip()
    for atributo in ("value_date", "value_string", "value_number"):
        valor = getattr(campo, atributo, None)
        if valor is not None:
            return str(valor)
    return campo.content

CAMPOS = ["VendorName", "VendorTaxId", "VendorAddress", "CustomerName", "CustomerTaxId", "CustomerAddress",
          "InvoiceId", "InvoiceDate", "DueDate", "PurchaseOrder", "SubTotal", "TotalTax", "InvoiceTotal", "PaymentTerm"]

campos_factura = {}
for idx, invoice in enumerate(invoices.documents or [], start=1):
    print(f"-------- Factura #{idx} (confianza del documento: {invoice.confidence}) --------")
    for nombre in CAMPOS:
        campo = invoice.fields.get(nombre)
        if campo:
            campos_factura[nombre] = {"valor": valor_campo(campo), "confianza": campo.confidence}
            print(f"{nombre:<16} {str(campos_factura[nombre]['valor']):<55} conf: {campo.confidence}")
        else:
            print(f"{nombre:<16} (no detectado)")

    campo_items = invoice.fields.get("Items")
    items = (campo_items.value_array if campo_items else None) or []
    print(f"\nÍtems detectados: {len(items)}")
    lineas_factura = []
    for i, item in enumerate(items, start=1):
        obj = item.value_object or {}
        linea = {k: valor_campo(obj.get(k)) for k in ("Description", "Quantity", "UnitPrice", "Amount")}
        lineas_factura.append(linea)
        print(f"  {i}. {linea['Description']} | cant: {linea['Quantity']} | p.u.: {linea['UnitPrice']} | importe: {linea['Amount']}")

-------- Factura #1 (confianza del documento: 1.0) --------
VendorName       Plásticos y Seguridad Andes S.A.C.                      conf: 0.913
VendorTaxId      20999888771                                             conf: 0.95
VendorAddress    Av. Los Inventados 742, Lima, Perú                      conf: 0.894
CustomerName     Banco Andino Ficticio S.A.                              conf: 0.919
CustomerTaxId    20111222333                                             conf: 0.948
CustomerAddress  Jr. Imaginario 1200, San Isidro, Lima                   conf: 0.893
InvoiceId        F001-004587                                             conf: 0.959
InvoiceDate      2026-10-15                                              conf: 0.95
DueDate          2026-11-14                                              conf: 0.95
PurchaseOrder    OC-2026-0931                                            conf: 0.933
SubTotal         S/ 40575.0                                              conf: 0.938
TotalTax

In [10]:
# Validación rápida: ¿el total extraído coincide con el esperado del documento sintético?
TOTAL_ESPERADO = 47878.50
campo_total = invoices.documents[0].fields.get("InvoiceTotal") if invoices.documents else None
monto = campo_total.value_currency.amount if campo_total and campo_total.value_currency else None
print("Total extraído:", monto, "| esperado:", TOTAL_ESPERADO,
      "| coincide:", monto is not None and abs(monto - TOTAL_ESPERADO) < 0.01)

Total extraído: 47878.5 | esperado: 47878.5 | coincide: True


## Paso 6. Guardar el resultado en un JSON indexable por Azure AI Search
La ficha se divide en **un registro por sección** (usando los encabezados del Markdown de `prebuilt-layout`): es la unidad
que el Copiloto recuperará con RAG y permite citar sección y página. La factura se guarda como un registro con sus campos.

Campos: `id`, `titulo`, `contenido` (mínimos) + metadatos `categoria`, `tipo_documento`, `producto`, `pagina`, `fuente`, `modelo`, `fecha_procesamiento`, `sintetico`.

In [11]:
import re

def pagina_de_offset(resultado, offset):
    for pg in resultado.pages or []:
        for span in pg.spans or []:
            if span.offset <= offset < span.offset + span.length:
                return pg.page_number
    return None

def secciones_markdown(resultado):
    """Corta el Markdown en bloques por encabezado (#, ##, ###)."""
    contenido = resultado.content or ""
    marcas = [m for m in re.finditer(r"^#{1,3} .+$", contenido, flags=re.M)]
    if not marcas:
        return [("Documento", contenido, 0)]
    bloques = []
    for i, m in enumerate(marcas):
        fin = marcas[i + 1].start() if i + 1 < len(marcas) else len(contenido)
        titulo = m.group(0).lstrip("#").strip()
        texto = contenido[m.end():fin].strip()
        if texto:
            bloques.append((titulo, texto, m.start()))
    return bloques

hoy = date.today().isoformat()
registros = []
for n, (titulo, texto, offset) in enumerate(secciones_markdown(layout), start=1):
    registros.append({
        "id": f"ficha-tc-condor-sec-{n:02d}",
        "titulo": titulo,
        "contenido": texto,
        "categoria": "tarjetas_de_credito",
        "tipo_documento": "ficha_producto_guia_atencion",
        "producto": "Tarjetas Cóndor",
        "pagina": pagina_de_offset(layout, offset),
        "fuente": RUTA_FICHA.name,
        "modelo": layout.model_id,
        "fecha_procesamiento": hoy,
        "sintetico": True,
    })

registros.append({
    "id": "factura-f001-004587",
    "titulo": f"Factura {campos_factura.get('InvoiceId', {}).get('valor')} - {campos_factura.get('VendorName', {}).get('valor')}",
    "contenido": invoices.content,
    "categoria": "facturas_proveedores",
    "tipo_documento": "factura",
    "producto": None,
    "pagina": 1,
    "fuente": RUTA_FACTURA.name,
    "modelo": invoices.model_id,
    "fecha_procesamiento": hoy,
    "sintetico": True,
    "campos_extraidos": campos_factura,
    "items": lineas_factura,
})

ruta_json = SALIDAS / "lab01_docintell_indexable.json"
ruta_json.write_text(json.dumps(registros, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(f"{len(registros)} registros guardados en {ruta_json}")
for r in registros:
    print(f"- {r['id']:<26} pág. {r['pagina']} | {r['titulo'][:60]} | {len(r['contenido'] or '')} caracteres")

10 registros guardados en outputs\lab01_docintell_indexable.json
- ficha-tc-condor-sec-01     pág. 1 | Banco Andino Ficticio S.A. Ficha de producto y guía de atenc | 139 caracteres
- ficha-tc-condor-sec-02     pág. 1 | 1. Propósito y alcance | 358 caracteres
- ficha-tc-condor-sec-03     pág. 1 | 2. Catálogo de tarjetas | 716 caracteres
- ficha-tc-condor-sec-04     pág. 1 | 3. Costos y tarifario referencial | 1315 caracteres
- ficha-tc-condor-sec-05     pág. 2 | 4. Requisitos de solicitud | 690 caracteres
- ficha-tc-condor-sec-06     pág. 2 | 5. Procedimiento de solicitud (canal agencia) | 593 caracteres
- ficha-tc-condor-sec-07     pág. 2 | 6. Bloqueo por pérdida o robo | 296 caracteres
- ficha-tc-condor-sec-08     pág. 2 | 7. Reclamos y plazos de atención | 549 caracteres
- ficha-tc-condor-sec-09     pág. 2 | 8. Restricciones y buenas prácticas | 569 caracteres
- factura-f001-004587        pág. 1 | Factura F001-004587 - Plásticos y Seguridad Andes S.A.C. | 1214 caracteres


In [12]:
# Evidencia adicional: Markdown completo de la ficha (útil para revisar el chunking)
(SALIDAS / "lab01_ficha_layout.md").write_text(layout.content, encoding="utf-8")
print("Markdown de layout guardado en outputs/lab01_ficha_layout.md")

Markdown de layout guardado en outputs/lab01_ficha_layout.md


## Checkpoint B
- [ ] Modelos listados, con `prebuilt-layout` y `prebuilt-invoice` disponibles.
- [ ] `prebuilt-layout`: páginas, texto, secciones y tablas revisados.
- [ ] `prebuilt-invoice`: campos, ítems y total validados.
- [ ] JSON indexable guardado en `outputs/lab01_docintell_indexable.json`.

## Aplicación al Proyecto Integrador
**¿Los documentos que utilizaría el Copiloto contienen tablas, formularios, documentos escaneados u otros elementos que justifiquen Document Intelligence?**

Sí. Las fichas de producto y guías de atención bancarias concentran la información clave en **tablas** (tarifario por tipo de tarjeta,
requisitos por tipo de cliente, plazos de reclamos) y en **listas de pasos**. Una extracción de texto plano pierde la relación fila-columna
(por ejemplo, qué membresía corresponde a qué tarjeta), lo que provocaría respuestas incorrectas del Copiloto. `prebuilt-layout` conserva
esa estructura, detecta títulos y secciones (base para un chunking por sección con cita de página) y también procesa documentos
**escaneados** (OCR), comunes en formularios firmados, hojas resumen o contratos.

Información que esperamos extraer: tarifarios (membresías, TEA, comisiones), requisitos por segmento, pasos de procedimientos
(solicitud, bloqueo, reclamos), plazos y restricciones. `prebuilt-invoice` es útil solo si el caso incorpora sustentos de pago o
facturas; para el Copiloto, el modelo central es `prebuilt-layout`.

## Reflexión
- **Capacidades útiles:** salida en Markdown con tablas, roles de párrafo (`title`, `sectionHeading`) y número de página por fragmento;
  campos tipados con nivel de confianza en facturas.
- **Cómo aplicarlas:** pipeline de ingesta *PDF → layout (Markdown) → chunks por sección → JSON → índice de Azure AI Search*, guardando
  página y sección para que cada respuesta del Copiloto cite su fuente.
- **Qué seguir validando:** tamaño óptimo de chunk cuando una sección es muy larga, comportamiento con tablas que cruzan páginas y con
  documentos escaneados de baja calidad, uso de umbrales de confianza para campos críticos y costo/latencia por página en un corpus mayor.